# GA4 会话与结账漏斗审计

步骤 9 的可复现、只读审计伴侣。

## tl;dr

在 47,911 个观察到的会话中，1,911 个进入结账；578 个在同一会话完成购买事件（30.25%），176 个形成同会话规范订单（9.21%）。`add_to_cart` 只有 1 个会话，因此不能作为主漏斗步骤。

## Context & Methods

### Key Assumptions

- 会话键为 `user_pseudo_id + ga_session_id`，时间按导出的 UTC 值。
- 主漏斗为 `begin_checkout` 后的同会话 `purchase`，并另报同会话规范订单。
- 规范订单排除 `(not set)` 和 `unknown` 交易号。
- 此为公开、匿名化样例的事件链路，不可解释为真实业务转化率。

## Data

来源批次：`9a2d8d7c-0f5b-4c57-a54d-98c59b9c5a27`；构建运行：`b6b845e1-0da3-4d90-97be-233d199790a5`；定义版本：`ga4_session_funnel_v1`。

In [1]:
from pathlib import Path
import os

import pandas as pd
import pymysql
from dotenv import load_dotenv

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(project_root / '.env')
batch_id = '9a2d8d7c-0f5b-4c57-a54d-98c59b9c5a27'
definition_version = 'ga4_session_funnel_v1'

connection = pymysql.connect(host=os.environ.get('MYSQL_HOST', '127.0.0.1'), port=int(os.environ.get('MYSQL_PORT', '3306')), user=os.environ.get('MYSQL_USER', 'root'), password=os.environ['MYSQL_ROOT_PASSWORD'], charset='utf8mb4')

## Results

In [2]:
funnel_summary = pd.read_sql(
    '''SELECT COUNT(*) AS observed_sessions, SUM(has_session_start) AS sessions_with_start,
              SUM(has_view_item) AS product_view_sessions, SUM(has_add_to_cart) AS add_to_cart_sessions,
              SUM(has_begin_checkout) AS checkout_start_sessions,
              SUM(has_checkout_to_purchase_event) AS checkout_to_purchase_event_sessions,
              SUM(has_checkout_to_canonical_order) AS checkout_to_canonical_order_sessions
       FROM insightops_mart.ga4_session_fact
       WHERE source_batch_id = %s AND definition_version = %s''',
    connection, params=(batch_id, definition_version)
)
funnel_summary

/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_5088/140847839.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  funnel_summary = pd.read_sql(


,observed_sessions,sessions_with_start,product_view_sessions,add_to_cart_sessions,checkout_start_sessions,checkout_to_purchase_event_sessions,checkout_to_canonical_order_sessions
0,47911,47302.0,11133.0,1.0,1911.0,578.0,176.0


In [3]:
daily_metrics = pd.read_sql(
    '''SELECT metric_date, observed_sessions, checkout_start_sessions,
              checkout_to_purchase_event_sessions, checkout_to_purchase_event_rate,
              checkout_to_canonical_order_sessions, checkout_to_canonical_order_rate
       FROM insightops_mart.ga4_daily_funnel_metric
       WHERE source_batch_id = %s AND definition_version = %s
       ORDER BY metric_date''',
    connection, params=(batch_id, definition_version)
)
daily_metrics

/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_5088/248071502.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  daily_metrics = pd.read_sql(


,metric_date,observed_sessions,checkout_start_sessions,checkout_to_purchase_event_sessions,checkout_to_purchase_event_rate,checkout_to_canonical_order_sessions,checkout_to_canonical_order_rate
0,2020-11-01,2625,95,13,0.136842,0,0.000000
1,2020-11-02,3726,130,40,0.307692,0,0.000000
2,2020-11-03,5347,144,48,0.333333,0,0.000000
3,2020-11-04,4325,118,28,0.237288,0,0.000000
4,2020-11-05,3840,127,26,0.204724,0,0.000000
5,2020-11-06,3319,154,48,0.311688,0,0.000000
6,2020-11-07,2271,111,28,0.252252,0,0.000000
7,2020-11-08,2163,100,31,0.310000,0,0.000000
8,2020-11-09,3271,135,43,0.318519,0,0.000000
9,2020-11-10,3824,173,62,0.358382,0,0.000000


## Takeaways

后续异常检测可使用日粒度的两条结账转化率；必须同时显示订单规范化口径。`add_to_cart` 稀疏性是数据覆盖限制，而非业务漏斗表现。